# 🏴‍☠️ JackSparrow Trading Agent — Colab Training Lab (v5)

**Expanded ~122 features (canonical + candlestick + chart patterns), TP/SL labeling, 15m–4h**

### Pipeline
```
Delta Exchange API  →  Candle Fetch  →  Feature Engineering
→  Label Generation  →  Train (Entry + Exit per TF)
→  Walk-Forward Validation  →  Export ZIP  →  Download
```

| Section | Description |
|---------|-------------|
| 0 | Environment setup & imports |
| 1 | Configuration |
| 2 | Candle download |
| 3 | Feature engineering |
| 4 | Label generation |
| 5 | Model training |
| 6 | Walk-forward validation |
| 7 | Export & download |


---
## 0 — Environment Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {sys.version}')


In [ ]:
!pip install -q xgboost lightgbm scikit-learn pandas numpy scipy joblib tqdm plotly matplotlib seaborn requests
print('✅ Dependencies installed (incl. scipy for chart patterns).')


In [ ]:
# ── 0.1  Clone repo (Colab) or use local path ─────────────────────────────────
import subprocess
from pathlib import Path
if IN_COLAB:
    REPO_URL = 'https://github.com/YOUR_ORG/trading-agent-2.git'  # Replace with your repo URL
    REPO_DIR = Path('/content/trading-agent-2')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    sys.path.insert(0, str(REPO_DIR))
    BASE = REPO_DIR
    print(f'✅ Repo at {REPO_DIR}')
else:
    REPO_DIR = Path.cwd()
    for _ in range(5):
        if (REPO_DIR / 'feature_store').exists():
            break
        REPO_DIR = REPO_DIR.parent
    sys.path.insert(0, str(REPO_DIR))
    BASE = REPO_DIR
    print(f'✅ Local path: {BASE}')

In [ ]:
import warnings, json, hashlib, time, threading, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
import lightgbm as lgb

from feature_store.unified_feature_engine import UnifiedFeatureEngine
from feature_store.feature_registry import EXPANDED_FEATURE_LIST

warnings.filterwarnings('ignore')
print('✅ Imports OK (UnifiedFeatureEngine + EXPANDED_FEATURE_LIST).')


---
## 1 — Configuration

In [ ]:
# ── 1.1  Project directories ─────────────────────────────────────────────────
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = Path('/content/drive/MyDrive/JackSparrow')
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f'✅ Drive mounted → {DRIVE_ROOT}')
    except Exception as e:
        print(f'⚠️  Drive mount skipped: {e}')
        DRIVE_ROOT = None
else:
    DRIVE_ROOT = None

try:
    _ = BASE
except NameError:
    BASE = Path('/content/trading-agent') if IN_COLAB else Path('.')

DIRS = [
    BASE / 'models',
    BASE / 'data' / 'candles',
    BASE / 'reports',
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DIR  = BASE / 'models'
DATA_DIR   = BASE / 'data' / 'candles'
REPORT_DIR = BASE / 'reports'

print('Directories:')
for d in DIRS:
    print(f'  ✓ {d}')


In [ ]:
# ── 1.2  Main configuration ───────────────────────────────────────────────────
@dataclass
class Config:
    symbol:            str   = 'BTCUSD'
    timeframes:        list  = field(default_factory=lambda: ['15m', '30m', '1h', '2h', '4h'])
    total_candles:     int   = 10_000   # candles to fetch per TF
    entry_threshold:   float = 0.003    # ±0.3 % forward return → BUY/SELL
    n_folds:           int   = 5
    train_split:       float = 0.70
    val_split:         float = 0.15     # remaining 15 % = test
    random_seed:       int   = 42
    n_jobs:            int   = -1
    top_n_features:    int   = 20       # use top-N features by variance

    # Adaptive lookahead (candles ahead for entry label)
    entry_lookahead_map: dict = field(default_factory=lambda: {
        '15m': 4,   # 1 hour ahead
        '30m': 2,   # 1 hour ahead
        '1h':  1,
        '2h':  1,
        '4h':  1,
    })
    # Exit label parameters
    exit_lookahead:   int   = 6
    exit_loss_pct:    float = 0.015
    exit_profit_pct:  float = 0.030

CFG = Config()
print('Configuration:')
for k, v in asdict(CFG).items():
    print(f'  {k:<25} = {v}')


---
## 2 — Candle Download

In [ ]:
# ── 2.1  API credentials ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['DELTA_API_KEY']    = userdata.get('DELTA_API_KEY')
    os.environ['DELTA_API_SECRET'] = userdata.get('DELTA_API_SECRET')
    print('✅ API keys loaded from Colab Secrets.')
except Exception:
    os.environ.setdefault('DELTA_API_KEY',    '')
    os.environ.setdefault('DELTA_API_SECRET', '')
    print('⚠️  No API keys — using public endpoint (candles only).')


In [ ]:
# ── 2.2  Rate limiter ──────────────────────────────────────────────────────────
class _TokenBucket:
    def __init__(self, rate=18.0, capacity=18.0):
        self._rate = rate; self._capacity = capacity
        self._tokens = capacity; self._last = time.monotonic()
        self._lock = threading.Lock()
    def acquire(self):
        with self._lock:
            now = time.monotonic()
            self._tokens = min(self._capacity, self._tokens + (now - self._last) * self._rate)
            self._last = now
            if self._tokens < 1.0:
                time.sleep((1.0 - self._tokens) / self._rate)
                self._tokens = 0.0
            else:
                self._tokens -= 1.0

_LIMITER = _TokenBucket()
print('✅ Rate limiter ready (18 req/s).')


In [ ]:
# ── 2.3  Candle fetcher ───────────────────────────────────────────────────────
import requests

DELTA_BASE = 'https://api.india.delta.exchange'
_TF_SECS   = {
    '1m':60, '3m':180, '5m':300, '15m':900, '30m':1800,
    '1h':3600, '2h':7200, '4h':14400, '1d':86400
}

def fetch_candles(symbol: str, resolution: str, total: int,
                  api_key: str = '', api_secret: str = '') -> pd.DataFrame:
    """Fetch candles from Delta Exchange with pagination, retries and gap handling."""
    all_candles = []
    headers     = {'api-key': api_key} if api_key else {}
    secs        = _TF_SECS.get(resolution, 900)
    end_time    = int(time.time())
    start_time  = end_time - secs * total

    with tqdm(total=total, desc=f'{symbol}/{resolution}', unit='candles') as pbar:
        cur = start_time
        while cur < end_time:
            _LIMITER.acquire()
            chunk_end = min(cur + 86_400, end_time)
            params = {'symbol': symbol, 'resolution': resolution, 'start': cur, 'end': chunk_end}

            data = []
            for attempt in range(5):
                try:
                    resp = requests.get(f'{DELTA_BASE}/v2/history/candles',
                                        params=params, headers=headers, timeout=15)
                    resp.raise_for_status()
                    data = resp.json().get('result', [])
                    break
                except requests.exceptions.HTTPError as e:
                    wait = 2 ** attempt
                    if e.response.status_code == 429:
                        time.sleep(wait)
                    elif attempt < 4:
                        time.sleep(wait)
                    else:
                        print(f'  ⚠ API error after 5 retries: {e}')
                except Exception as e:
                    if attempt < 4:
                        time.sleep(2 ** attempt)
                    else:
                        print(f'  ⚠ Connection error: {e}')

            if data:
                all_candles.extend(data)
                pbar.update(len(data))
            cur = chunk_end

    if not all_candles:
        raise ValueError(f'No candles returned for {symbol}/{resolution}')

    df = pd.DataFrame(all_candles)
    df = df.rename(columns={'time': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    for col in ['open','high','low','close','volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = (df.sort_values('timestamp')
            .drop_duplicates('timestamp')
            .dropna(subset=['open','high','low','close','volume'])
            .reset_index(drop=True))

    # Use largest continuous segment if big gaps exist
    gaps = df['timestamp'].diff().dt.total_seconds()
    bad  = (gaps > secs * 2).sum()
    if bad:
        print(f'  ⚠ {bad} gaps detected — using largest continuous segment')
        split_pts = gaps[gaps > secs * 2].index.tolist()
        segs  = []
        prev  = 0
        for pt in split_pts:
            segs.append(df.iloc[prev:pt])
            prev = pt
        segs.append(df.iloc[prev:])
        df = max(segs, key=len).reset_index(drop=True)

    print(f'  ✅ {len(df)} candles  {str(df["timestamp"].iloc[0])[:19]}  →  {str(df["timestamp"].iloc[-1])[:19]}')
    return df

print('✅ Candle fetcher defined.')


In [ ]:
# ── 2.4  Download / load from cache ──────────────────────────────────────────
api_key    = os.environ.get('DELTA_API_KEY', '')
api_secret = os.environ.get('DELTA_API_SECRET', '')

RAW: Dict[str, pd.DataFrame] = {}
for tf in CFG.timeframes:
    cache = DATA_DIR / f'{CFG.symbol}_{tf}.parquet'
    if cache.exists():
        RAW[tf] = pd.read_parquet(cache)
        print(f'  [{tf}] Loaded from cache ({len(RAW[tf])} candles)')
    else:
        RAW[tf] = fetch_candles(CFG.symbol, tf, CFG.total_candles, api_key, api_secret)
        RAW[tf].to_parquet(cache, index=False)
        print(f'  [{tf}] Saved to cache')

print(f'\n✅ Loaded: {list(RAW.keys())}')


In [ ]:
# ── 2.5  Quick overview ──────────────────────────────────────────────────────
rows = []
for tf, df in RAW.items():
    rows.append({
        'timeframe': tf,
        'candles':   len(df),
        'from':      str(df['timestamp'].iloc[0])[:19],
        'to':        str(df['timestamp'].iloc[-1])[:19],
        'price_min': round(df['close'].min(), 2),
        'price_max': round(df['close'].max(), 2),
    })
pd.DataFrame(rows)


---
## 3 — Feature Engineering

In [ ]:
# ── 3.1  Feature computation (UnifiedFeatureEngine, expanded ~122 features) ─────
TF_TO_RESOLUTION = {'15m': 15, '30m': 30, '1h': 60, '2h': 120, '4h': 240}

engine = UnifiedFeatureEngine()

def compute_features(df: pd.DataFrame, resolution_minutes: int = 60) -> pd.DataFrame:
    """Compute all features via UnifiedFeatureEngine (canonical + candlestick + chart patterns)."""
    return engine.compute_batch(
        df, resolution_minutes=resolution_minutes,
        include_pattern_features=True, fill_invalid=True
    )

print(f'✅ UnifiedFeatureEngine ready. EXPANDED_FEATURE_LIST: {len(EXPANDED_FEATURE_LIST)} features.')


In [ ]:
# ── 3.2  Apply to all timeframes ─────────────────────────────────────────────
WARMUP = 200   # drop first N rows (pattern engines need ~100 candles)
FEATS:  Dict[str, pd.DataFrame] = {}
PRICES: Dict[str, pd.DataFrame] = {}   # aligned raw OHLCV

for tf in CFG.timeframes:
    raw = RAW[tf].copy()
    res_min = TF_TO_RESOLUTION.get(tf, 60)
    feat = compute_features(raw, resolution_minutes=res_min).iloc[WARMUP:].reset_index(drop=True)
    aligned_raw = raw.iloc[WARMUP:].reset_index(drop=True)

    # Ensure column order matches EXPANDED_FEATURE_LIST (train-serve parity)
    feat = feat[[c for c in EXPANDED_FEATURE_LIST if c in feat.columns]]

    FEATS[tf]  = feat
    PRICES[tf] = aligned_raw

    print(f'  [{tf}]  {len(feat)} rows × {feat.shape[1]} features')

FEATURE_NAMES = list(EXPANDED_FEATURE_LIST)  # definitive order for export
print(f'\n✅ Feature matrices computed.  Features: {len(FEATURE_NAMES)} (expanded schema)')


---
## 4 (Alternative) — TP/SL Outcome Labeling

**Optional:** Use trade-outcome-based labels instead of next-bar return. Simulates entry at close[t], checks if TP (1.5%) or SL (0.8%) is hit first within max_bars. Better aligns model with actual trading profitability.

```python
# Uncomment to use TP/SL labeling instead of make_entry_labels:
# from scripts.training_utils import create_trade_outcome_labels
# entry_labels = create_trade_outcome_labels(PRICES[tf], tp_pct=0.015, sl_pct=0.008, max_bars=20)
```

In [ ]:
# TP/SL outcome labeling (alternative to make_entry_labels)
def create_trade_outcome_labels(df, tp_pct=0.015, sl_pct=0.008, max_bars=20):
    """Label by simulated trade: TP hit first=BUY(2), SL first=SELL(0), else HOLD(1)."""
    closes, highs, lows = df['close'].values, df['high'].values, df['low'].values
    n = len(df)
    labels = []
    for i in range(n - max_bars):
        entry, tp_long, sl_long = closes[i], closes[i] * (1 + tp_pct), closes[i] * (1 - sl_pct)
        label = 1
        for j in range(1, max_bars + 1):
            if highs[i + j] >= tp_long: label = 2; break
            if lows[i + j] <= sl_long: label = 0; break
        labels.append(label)
    labels.extend([1] * max_bars)
    return pd.Series(labels, index=df.index)

# Class weight for XGBoost (use in XGBClassifier)
from sklearn.utils.class_weight import compute_class_weight
def get_scale_pos_weight(y, pos_class=2):
    w = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return sum(w[c] for c in np.unique(y) if c != pos_class) / w[pos_class]

In [ ]:
# ── 3.3  Feature set (expanded schema, no selection) ──────────────────────────
# Use full EXPANDED_FEATURE_LIST for train-serve parity. Top-N selection disabled.
TOP_FEATURES: Dict[str, List[str]] = {tf: list(FEATS[tf].columns) for tf in CFG.timeframes}

# Top feature importances will be logged after training
tf_show = '1h' if '1h' in CFG.timeframes else CFG.timeframes[0]
print(f'Using {len(TOP_FEATURES[tf_show])} features for {tf_show} (expanded schema)')


---
## 4 — Label Generation

In [ ]:
# ── 4.1  Entry labels: SELL=0, HOLD=1, BUY=2 ─────────────────────────────────
def make_entry_labels(close: pd.Series, lookahead: int = 1,
                      threshold: float = 0.003) -> pd.Series:
    fwd = close.shift(-max(lookahead, 1)) / close - 1.0
    lbl = np.where(fwd > threshold, 2,
          np.where(fwd < -threshold, 0, 1))
    return pd.Series(lbl, index=close.index, dtype=int)


# ── 4.2  Exit labels: 0=hold, 1=exit ─────────────────────────────────────────
def make_exit_labels(close: pd.Series, lookahead: int = 6,
                     loss_pct: float = 0.015,
                     profit_pct: float = 0.030) -> pd.Series:
    arr = close.values
    n   = len(arr)
    lbl = np.zeros(n, dtype=int)
    for i in range(n - lookahead):
        entry  = arr[i]
        future = arr[i+1: i+1+lookahead]
        if (entry - future.min()) / (entry + 1e-10) >= loss_pct:
            lbl[i] = 1
        elif (future.max() - entry) / (entry + 1e-10) >= profit_pct:
            lbl[i] = 1
    return pd.Series(lbl, index=close.index, dtype=int)


print('✅ Label functions defined.')


In [ ]:
# ── 4.3  Generate labels for all timeframes ───────────────────────────────────
# Entry: TP/SL outcome labeling (BUY=2 if TP hit first, SELL=0 if SL first, else HOLD=1)
# Exit: binary (hold=0, exit=1 when TP or SL hit within lookahead)
ENTRY_LABELS: Dict[str, pd.Series] = {}
EXIT_LABELS:  Dict[str, pd.Series] = {}

TP_PCT, SL_PCT, MAX_BARS = 0.015, 0.008, 20

for tf in CFG.timeframes:
    prices_df = PRICES[tf].reset_index(drop=True)
    n_use = len(FEATS[tf])

    close_v = prices_df['close'].iloc[:n_use]

    # TP/SL entry labels
    ey = create_trade_outcome_labels(prices_df.iloc[:n_use], tp_pct=TP_PCT, sl_pct=SL_PCT, max_bars=MAX_BARS)
    ey = ey.iloc[:n_use].reset_index(drop=True)

    # Exit labels (unchanged)
    xy = make_exit_labels(close_v, lookahead=CFG.exit_lookahead,
                          loss_pct=CFG.exit_loss_pct, profit_pct=CFG.exit_profit_pct)

    safe_n = n_use - max(MAX_BARS, CFG.exit_lookahead)
    ENTRY_LABELS[tf] = ey.iloc[:safe_n].reset_index(drop=True)
    EXIT_LABELS[tf]  = xy.iloc[:safe_n].reset_index(drop=True)

    el = ENTRY_LABELS[tf]
    xl = EXIT_LABELS[tf]
    print(
        f'  [{tf}]  samples={len(el)}  '
        f'SELL={int((el==0).sum())}  HOLD={int((el==1).sum())}  BUY={int((el==2).sum())}  '
        f'exit_pos={int(xl.sum())}  TP_SL={TP_PCT:.2%}/{SL_PCT:.2%}'
    )

print('\n✅ Labels generated (TP/SL entry, binary exit).')


In [ ]:
# ── 4.4  Label distribution chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tf_show = '1h' if '1h' in CFG.timeframes else CFG.timeframes[0]

# Entry
ev = ENTRY_LABELS[tf_show].value_counts().sort_index()
axes[0].bar(['SELL','HOLD','BUY'], ev.values,
            color=['#e74c3c','#95a5a6','#2ecc71'])
axes[0].set_title(f'Entry Labels — {tf_show}')
axes[0].set_ylabel('Count')

# Exit
xv = EXIT_LABELS[tf_show].value_counts().sort_index()
axes[1].bar(['HOLD','EXIT'], xv.values, color=['#3498db','#e74c3c'])
axes[1].set_title(f'Exit Labels — {tf_show}')

plt.suptitle(f'{CFG.symbol} Label Distributions')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'label_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Chart saved.')


---
## 5 — Model Training

One **entry model** (3-class XGBoost) + one **exit model** (binary XGBoost) per timeframe. Simple and effective.

In [ ]:
# ── 5.1  Model builders ───────────────────────────────────────────────────────
def make_entry_model(seed: int, scale_pos_weight=None) -> xgb.XGBClassifier:
    """3-class entry signal classifier."""    return xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', use_label_encoder=False,
        early_stopping_rounds=30,
        random_state=seed, n_jobs=1, verbosity=0,
    )

def make_exit_model(seed: int, scale_pos_weight: float = 1.0) -> xgb.XGBClassifier:
    """Binary exit classifier."""    return xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        objective='binary:logistic', eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False, early_stopping_rounds=30,
        random_state=seed, n_jobs=1, verbosity=0,
    )

print('✅ Model builders defined.')


In [ ]:
# ── 5.2  Train entry + exit model for every timeframe ─────────────────────────
ENTRY_MODELS:   Dict[str, Any] = {}
EXIT_MODELS:    Dict[str, Any] = {}
ENTRY_SCALERS:  Dict[str, Any] = {}
EXIT_SCALERS:   Dict[str, Any] = {}
TRAIN_METRICS:  Dict[str, Dict] = {}

for tf in CFG.timeframes:
    print(f'\n── [{tf}] ──────────────────────────────────────────────')

    top_feats = TOP_FEATURES[tf]
    n_safe    = len(ENTRY_LABELS[tf])
    X_all     = FEATS[tf][top_feats].iloc[:n_safe].values.astype(np.float32)
    y_entry   = ENTRY_LABELS[tf].values
    y_exit    = EXIT_LABELS[tf].values

    n      = len(X_all)
    tr_end = int(n * CFG.train_split)
    va_end = int(n * (CFG.train_split + CFG.val_split))

    X_tr, y_e_tr = X_all[:tr_end], y_entry[:tr_end]
    X_va, y_e_va = X_all[tr_end:va_end], y_entry[tr_end:va_end]
    X_te, y_e_te = X_all[va_end:], y_entry[va_end:]
    y_x_tr = y_exit[:tr_end];  y_x_te = y_exit[va_end:]

    # ── Scale ────────────────────────────────────────────────────────────
    e_scaler = RobustScaler().fit(X_tr)
    X_tr_s   = e_scaler.transform(X_tr)
    X_va_s   = e_scaler.transform(X_va)
    X_te_s   = e_scaler.transform(X_te)

    # ── Entry model (with sample weights for TP/SL class imbalance) ───────
    sw_tr = compute_sample_weight('balanced', y_e_tr)
    e_model = make_entry_model(CFG.random_seed)
    e_model.fit(
        X_tr_s, y_e_tr, sample_weight=sw_tr,
        eval_set=[(X_va_s, y_e_va)],
        verbose=False
    )
    y_e_pred = e_model.predict(X_te_s)
    e_acc    = accuracy_score(y_e_te, y_e_pred)
    e_f1     = f1_score(y_e_te, y_e_pred, average='macro', zero_division=0)
    print(f'  Entry  →  acc={e_acc:.4f}  f1_macro={e_f1:.4f}  (n_test={len(y_e_te)})')

    ENTRY_MODELS[tf]  = e_model
    ENTRY_SCALERS[tf] = e_scaler

    # ── Exit model ───────────────────────────────────────────────────────
    n0, n1 = (y_x_tr == 0).sum(), (y_x_tr == 1).sum()
    spw    = n0 / max(n1, 1)

    x_scaler = RobustScaler().fit(X_tr)
    X_tr_xs  = x_scaler.transform(X_tr)
    X_va_xs  = x_scaler.transform(X_va)
    X_te_xs  = x_scaler.transform(X_te)

    x_model = make_exit_model(CFG.random_seed, scale_pos_weight=spw)
    x_model.fit(
        X_tr_xs, y_x_tr,
        eval_set=[(X_va_xs, y_exit[tr_end:va_end])],
        verbose=False
    )
    y_x_pred = x_model.predict(X_te_xs)
    x_acc    = accuracy_score(y_x_te, y_x_pred)
    x_f1     = f1_score(y_x_te, y_x_pred, zero_division=0)
    print(f'  Exit   →  acc={x_acc:.4f}  f1={x_f1:.4f}  spw={spw:.2f}')

    EXIT_MODELS[tf]   = x_model
    EXIT_SCALERS[tf]  = x_scaler

    TRAIN_METRICS[tf] = {
        'entry_acc': round(e_acc, 4), 'entry_f1': round(e_f1, 4),
        'exit_acc':  round(x_acc, 4), 'exit_f1':  round(x_f1, 4),
        'n_train': tr_end, 'n_test': len(y_e_te),
    }

print('\n✅ All models trained.')


In [ ]:
# ── 5.3  Training summary ─────────────────────────────────────────────────────
rows = []
for tf, m in TRAIN_METRICS.items():
    rows.append({'TF': tf, **m})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


---
## 6 — Walk-Forward Validation

In [ ]:
# ── 6.1  TimeSeriesSplit walk-forward ─────────────────────────────────────────
WF_RESULTS: Dict[str, List[Dict]] = {}

for tf in CFG.timeframes:
    print(f'\n  [{tf}]  walk-forward ({CFG.n_folds} folds) …')

    top_feats = TOP_FEATURES[tf]
    n_safe    = len(ENTRY_LABELS[tf])
    X_all     = FEATS[tf][top_feats].iloc[:n_safe].values.astype(np.float32)
    y_entry   = ENTRY_LABELS[tf].values
    close_arr = PRICES[tf]['close'].iloc[:n_safe].values

    tscv   = TimeSeriesSplit(n_splits=CFG.n_folds)
    folds  = []

    for fold, (tri, tei) in enumerate(tscv.split(X_all)):
        X_tr, X_te = X_all[tri], X_all[tei]
        y_tr, y_te = y_entry[tri], y_entry[tei]

        scaler = RobustScaler().fit(X_tr)
        X_tr_s = scaler.transform(X_tr)
        X_te_s = scaler.transform(X_te)

        val_split = int(len(X_tr_s) * 0.9)
        m = xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective='multi:softprob', num_class=3,
            eval_metric='mlogloss', use_label_encoder=False,
            early_stopping_rounds=30,
            random_state=CFG.random_seed, n_jobs=-1, verbosity=0
        )
        m.fit(
            X_tr_s[:val_split], y_tr[:val_split],
            eval_set=[(X_tr_s[val_split:], y_tr[val_split:])],
            verbose=False
        )

        preds  = m.predict(X_te_s)
        acc    = accuracy_score(y_te, preds)
        f1_m   = f1_score(y_te, preds, average='macro', zero_division=0)

        # Sharpe proxy
        signal   = np.where(preds==2, 1.0, np.where(preds==0, -1.0, 0.0))
        c_te     = close_arr[tei]
        nh       = min(len(signal), len(c_te) - 1)
        fwd_ret  = np.log(c_te[1:nh+1] / c_te[:nh] + 1e-10)
        strat    = signal[:nh] * fwd_ret
        sharpe   = (strat.mean() / (strat.std() + 1e-10) * np.sqrt(252))

        folds.append({
            'fold': fold+1, 'n_train': len(tri), 'n_test': len(tei),
            'accuracy': round(acc, 4), 'f1_macro': round(f1_m, 4),
            'sharpe': round(float(sharpe), 4),
        })
        print(f'    Fold {fold+1}  acc={acc:.4f}  f1={f1_m:.4f}  sharpe={sharpe:.4f}')

    WF_RESULTS[tf] = folds

print('\n✅ Walk-forward validation complete.')


In [ ]:
# ── 6.2  Summary table + chart ───────────────────────────────────────────────
rows = []
for tf, folds in WF_RESULTS.items():
    for f in folds:
        rows.append({'timeframe': tf, **f})
wf_df = pd.DataFrame(rows)

summary_wf = (wf_df.groupby('timeframe')[['accuracy','f1_macro','sharpe']]
                   .mean().round(4).reset_index())
print('Walk-forward mean metrics:')
print(summary_wf.to_string(index=False))

fig = px.box(wf_df, x='timeframe', y='sharpe', color='timeframe',
             title='Walk-Forward Sharpe Distribution by Timeframe',
             points='all', template='plotly_dark')
fig.show()
fig.write_html(str(REPORT_DIR / 'walk_forward_sharpe.html'))
print('✅ Chart saved.')


In [ ]:
# ── 6.3  Feature importance plot (top 15 features per TF) ────────────────────
fig, axes = plt.subplots(1, len(CFG.timeframes), figsize=(5*len(CFG.timeframes), 5))
if len(CFG.timeframes) == 1:
    axes = [axes]

for ax, tf in zip(axes, CFG.timeframes):
    m     = ENTRY_MODELS[tf]
    feats = TOP_FEATURES[tf]
    imp   = pd.Series(m.feature_importances_, index=feats).nlargest(15)
    imp.plot(kind='barh', ax=ax, color='#3498db')
    ax.set_title(f'{tf} Entry')
    ax.set_xlabel('Importance')
    ax.invert_yaxis()

plt.suptitle(f'{CFG.symbol} Entry Model Feature Importance', y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved.')


In [ ]:
# ── 5.4  Feature importance sanity (pattern features) ─────────────────────────
PATTERN_PREFIXES = ('cdl_', 'chp_', 'sr_', 'tl_', 'bo_')
for tf in CFG.timeframes:
    m = ENTRY_MODELS[tf]
    feats = TOP_FEATURES[tf]
    imp = pd.Series(m.feature_importances_, index=feats)
    pattern_imp = imp[[f for f in feats if any(f.startswith(p) for p in PATTERN_PREFIXES)]]
    if len(pattern_imp) > 0:
        top_pat = pattern_imp.nlargest(5)
        print(f'  [{tf}] Top pattern features: {list(top_pat.index)}')
        if pattern_imp.sum() < 0.01:
            print(f'    ⚠️  Pattern features have near-zero total importance ({pattern_imp.sum():.4f})')

**Local storage:** After downloading the ZIP, extract its contents into:
`agent/model_storage/jacksparrow_v5_BTCUSD_YYYY-MM-DD/`

Then set `MODEL_DIR=./agent/model_storage/jacksparrow_v5_BTCUSD_YYYY-MM-DD` in `.env`.

---
## 7 — Export & Download

Saves all models + metadata and creates a downloadable ZIP.

In [ ]:
# ── 7.1  Dataset fingerprint ──────────────────────────────────────────────────
def sha256_df(df: pd.DataFrame) -> str:
    arr = df[['open','high','low','close','volume']].values.astype(np.float64)
    return hashlib.sha256(arr.tobytes()).hexdigest()[:16]

DATA_HASHES = {tf: sha256_df(RAW[tf]) for tf in CFG.timeframes}
for tf, h in DATA_HASHES.items():
    print(f'  [{tf}]  sha256={h}')
print('✅ Hashes computed.')


In [ ]:
# ── 7.2  Save models + metadata (versioned folder for local storage) ─────────
import sys
import sklearn, xgboost

# Versioned folder: extract to agent/model_storage/jacksparrow_v5_BTCUSD_YYYY-MM-DD
EXPORT_VERSION = '5.0.0'
EXPORT_DATE = datetime.now().strftime('%Y-%m-%d')
EXPORT_DIR = BASE / 'models' / f'jacksparrow_v5_{CFG.symbol}_{EXPORT_DATE}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Export folder: {EXPORT_DIR}')

for tf in CFG.timeframes:
    tag = f'{CFG.symbol}_{tf}'
    print(f'  Saving [{tag}] …')

    joblib.dump(ENTRY_MODELS[tf],   EXPORT_DIR / f'entry_model_{tag}.joblib')
    joblib.dump(ENTRY_SCALERS[tf],  EXPORT_DIR / f'entry_scaler_{tag}.joblib')
    joblib.dump(EXIT_MODELS[tf],    EXPORT_DIR / f'exit_model_{tag}.joblib')
    joblib.dump(EXIT_SCALERS[tf],   EXPORT_DIR / f'exit_scaler_{tag}.joblib')

    # Feature names: EXPANDED_FEATURE_LIST for train-serve parity
    with open(EXPORT_DIR / f'features_{tag}.json', 'w') as f:
        json.dump({'features': list(EXPANDED_FEATURE_LIST)}, f, indent=2)

    wf_mean = pd.DataFrame(WF_RESULTS[tf])[['accuracy','f1_macro','sharpe']].mean().round(4).to_dict()

    metadata = {
        'model_name':    f'jacksparrow_{tag}',
        'version':       EXPORT_VERSION,
        'symbol':        CFG.symbol,
        'timeframe':     tf,
        'trained_at':    datetime.now(timezone.utc).isoformat(),
        'dataset_sha256': DATA_HASHES[tf],
        'dataset_range': {
            'from':    str(RAW[tf]['timestamp'].iloc[0]),
            'to':      str(RAW[tf]['timestamp'].iloc[-1]),
            'candles': len(RAW[tf]),
        },
        'features':        list(EXPANDED_FEATURE_LIST),
        'n_features':      len(EXPANDED_FEATURE_LIST),
        'signal_classes':  {'entry': {'0': 'SELL', '1': 'HOLD', '2': 'BUY'},
                            'exit':  {'0': 'HOLD', '1': 'EXIT'}},
        'train_metrics':   TRAIN_METRICS[tf],
        'walkforward_mean': wf_mean,
        'config': {
            'entry_tp_pct':     TP_PCT,
            'entry_sl_pct':     SL_PCT,
            'entry_max_bars':   MAX_BARS,
            'exit_lookahead':   CFG.exit_lookahead,
            'exit_loss_pct':    CFG.exit_loss_pct,
            'exit_profit_pct':  CFG.exit_profit_pct,
        },
        'library_versions': {
            'python':   sys.version.split()[0],
            'xgboost':  xgboost.__version__,
            'sklearn':  sklearn.__version__,
            'numpy':    np.__version__,
            'pandas':   pd.__version__,
        },
        'artifacts': {
            'entry_model':  f'entry_model_{tag}.joblib',
            'entry_scaler': f'entry_scaler_{tag}.joblib',
            'exit_model':   f'exit_model_{tag}.joblib',
            'exit_scaler':  f'exit_scaler_{tag}.joblib',
            'features':     f'features_{tag}.json',
            'metadata':     f'metadata_{tag}.json',
        }
    }
    with open(EXPORT_DIR / f'metadata_{tag}.json', 'w') as f:
        json.dump(metadata, f, indent=2, default=str)

    n_files = len(list(EXPORT_DIR.glob(f'*{tag}*')))
    print(f'    ✅ {n_files} artifacts saved for [{tag}]')

print('\n✅ All models exported.')


In [ ]:
# ── 7.3  Validate saved artifacts ─────────────────────────────────────────────
print('Validating artifacts …\n')
errors = []
for f in sorted(EXPORT_DIR.iterdir()):
    try:
        if f.suffix == '.joblib':
            obj = joblib.load(f)
            print(f'  ✓ {f.name:<55}  {type(obj).__name__}')
        elif f.suffix == '.json':
            with open(f) as fh:
                d = json.load(fh)
            print(f'  ✓ {f.name:<55}  {len(d)} keys')
    except Exception as e:
        errors.append(f.name)
        print(f'  ✗ {f.name}: {e}')

status = '✅ All OK' if not errors else f'❌ Errors: {errors}'
print(f'\n{status}')


In [ ]:
# ── 7.4  Create ZIP and download ──────────────────────────────────────────────
zip_name = f'jacksparrow_models_v5_{CFG.symbol}_{EXPORT_DATE}.zip'
zip_path = BASE / zip_name

# Create zip of the versioned export folder
shutil.make_archive(str(BASE / zip_name.replace('.zip','')), 'zip', EXPORT_DIR)
print(f'✅ ZIP created: {zip_path}  ({zip_path.stat().st_size / 1024:.0f} KB)')

# Also backup to Google Drive if available
if DRIVE_ROOT is not None:
    import shutil as _sh
    _sh.copy(str(zip_path), str(DRIVE_ROOT / zip_name))
    print(f'✅ Backed up to Drive: {DRIVE_ROOT / zip_name}')

# ── Download (Colab only) ─────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
    print('✅ Download started.')
else:
    print(f'📦 ZIP ready at: {zip_path}')


In [ ]:
# ── 7.5  Training summary ─────────────────────────────────────────────────────
print('\n' + '='*70)
print('🏴‍☠️  JackSparrow v5 — Training Complete')
print('='*70)
print(f'Symbol:       {CFG.symbol}')
print(f'Timeframes:   {CFG.timeframes}')
print(f'Models saved: {len(list(EXPORT_DIR.glob("*.joblib")))} .joblib files')
print()
print('Per-timeframe results:')
print(f'  {"TF":<6} {"Entry Acc":>10} {"Entry F1":>10} {"Exit Acc":>10} {"WF Sharpe":>10}')
print(f'  {"─"*6} {"─"*10} {"─"*10} {"─"*10} {"─"*10}')
for tf in CFG.timeframes:
    m  = TRAIN_METRICS[tf]
    wf = pd.DataFrame(WF_RESULTS[tf])['sharpe'].mean()
    print(f'  {tf:<6} {m["entry_acc"]:>10.4f} {m["entry_f1"]:>10.4f} {m["exit_acc"]:>10.4f} {wf:>10.4f}')
print()
print(f'ZIP: {zip_path}')
print('='*70)
